# Titanic — Model Inference

Load the best model from MLflow Model Registry and generate `submission.csv`.

In [ ]:
!pip install mlflow category_encoders dagshub --quiet

In [ ]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import category_encoders as ce

warnings.filterwarnings('ignore')

## MLflow / DagsHub Setup

Must match the settings used in `model_experiment.ipynb`.

In [ ]:
USE_DAGSHUB = False

DAGSHUB_USERNAME = "YOUR_DAGSHUB_USERNAME"
DAGSHUB_REPO     = "YOUR_REPO_NAME"
DAGSHUB_TOKEN    = "YOUR_DAGSHUB_TOKEN"

if USE_DAGSHUB:
    os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
    os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
    tracking_uri = f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow"
    mlflow.set_tracking_uri(tracking_uri)
else:
    mlflow.set_tracking_uri("mlruns")

print("Tracking URI:", mlflow.get_tracking_uri())

## Load Model from Registry

In [ ]:
MODEL_NAME    = "titanic-best-model"
MODEL_VERSION = "latest"   # or a specific version number like "1"

model_uri = f"models:/{MODEL_NAME}/{MODEL_VERSION}"
model     = mlflow.sklearn.load_model(model_uri)
print("Model loaded:", type(model).__name__)

## Load Preprocessing Artifacts

In [ ]:
with open('preprocessing_artifacts.pkl', 'rb') as f:
    artifacts = pickle.load(f)

best_encoding = artifacts['best_encoding']
selected_ohe  = artifacts['selected_ohe']
selected_woe  = artifacts['selected_woe']
woe_encoder   = artifacts['woe_encoder']
numeric_cols  = artifacts['numeric_cols']
OHE_COLS      = artifacts['ohe_cols']
WOE_COLS      = artifacts['woe_cols']

print("Best encoding used:", best_encoding)

## Preprocessing Pipeline (mirrors model_experiment.ipynb)

In [ ]:
def extract_title(name: str) -> str:
    title = name.split(',')[1].split('.')[0].strip()
    rare = {'Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'}
    if title in rare:
        return 'Rare'
    if title == 'Mlle':
        return 'Miss'
    if title in ('Ms', 'Mme'):
        return 'Mrs'
    return title


def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['Title'] = df['Name'].apply(extract_title)
    age_medians = df.groupby(['Pclass', 'Sex'])['Age'].transform('median')
    df['Age']      = df['Age'].fillna(age_medians)
    df['Age']      = df['Age'].fillna(df['Age'].median())
    df['Fare']     = df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'))
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    df['Deck']     = df['Cabin'].str[0].fillna('Unknown')
    df.drop(columns=['Cabin', 'Name', 'Ticket'], errors='ignore', inplace=True)
    return df


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']    = (df['FamilySize'] == 1).astype(int)
    df['AgeBin']     = pd.cut(df['Age'],  bins=[0,12,18,35,60,120],
                              labels=['Child','Teen','Adult','MidAge','Senior'])
    df['FareBin']    = pd.qcut(df['Fare'], q=4,
                               labels=['Low','Mid','High','VHigh'], duplicates='drop')
    return df

## Load Test Data & Generate Submission

In [ ]:
test_raw = pd.read_csv("data/test.csv")
passenger_ids = test_raw['PassengerId'].copy()

test_clean = clean(test_raw)
test_fe    = add_features(test_clean)

if best_encoding == 'OHE':
    ohe_dummies = pd.get_dummies(test_fe[OHE_COLS], columns=OHE_COLS, drop_first=False)
    X_test = pd.concat([test_fe[numeric_cols].reset_index(drop=True), ohe_dummies], axis=1)

    for col in selected_ohe:
        if col not in X_test.columns:
            X_test[col] = 0
    X_test = X_test[selected_ohe]

else:
    test_for_woe = test_fe[numeric_cols + WOE_COLS].copy()
    for c in ['AgeBin', 'FareBin']:
        test_for_woe[c] = test_for_woe[c].astype(str)
    X_test_woe = woe_encoder.transform(test_for_woe)
    X_test     = X_test_woe[selected_woe]

print("Test feature matrix shape:", X_test.shape)
predictions = model.predict(X_test.values.astype(float))

submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived':    predictions
})
submission.to_csv('submission.csv', index=False)
print("submission.csv saved.")
submission.head(10)

In [ ]:
print("Survival rate in submission:", submission['Survived'].mean().round(4))
submission['Survived'].value_counts()